# UdaPlay: Persistent Game Retrieval

## Local Retrieval Layer

Create a persistent Chroma collection from 15 structured game records. Each
record is rendered as searchable text while its original fields remain
available as metadata. The final cell performs a one-result retrieval smoke test.

### Environment

Load the local API configuration and retrieval dependencies.

In [ ]:
import json
import os

import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from dotenv import load_dotenv

In [ ]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is required.")

### Persistent Chroma Client

Store embeddings locally so the collection remains available across sessions.

In [ ]:
chroma_client = chromadb.PersistentClient(path="chromadb")

### Game Collection

Use an explicit embedding model for consistent indexing and retrieval.

In [ ]:
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=OpenAIEmbeddingFunction(
        api_key_env_var="OPENAI_API_KEY",
        model_name="text-embedding-3-large"
    )
)

### Idempotent Ingestion

Use each source filename as a stable document ID. Upserts make the ingestion
cell safe to rerun without creating duplicate records.

In [ ]:
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = (
        f"[{game['Platform']}] {game['Name']} "
        f"({game['YearOfRelease']}) - {game['Description']}"
    )

    doc_id = os.path.splitext(file_name)[0]

    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Semantic Retrieval

Verify that the nearest local record directly addresses a vehicle-related query.

In [ ]:
question = 'List one of the games that feature cars.'
print(f'Question: {question}')
results = collection.query(query_texts=[question], n_results=1)
print('\n'.join(results['documents'][0]))